# Banking Customer 360 Analytics

## 03 - Data Integration

Purpose: Combine the cleaned banking tables into a unified Customer 360 dataset using the relationships identified during data discovery.

## 1. Load Cleaned Tables

Load the validated tables from the processed-data folder so they can be integrated into the Customer 360 dataset.


In [1]:
import pandas as pd
from pathlib import Path

processed_path = Path("../data/processed")

customers = pd.read_csv(processed_path / "customers.csv")
accounts = pd.read_csv(processed_path / "accounts.csv")
transactions = pd.read_csv(processed_path / "transactions.csv")
loans = pd.read_csv(processed_path / "loans.csv")
customer_service = pd.read_csv(processed_path / "customer_service.csv")

print("Customers:", customers.shape)
print("Accounts:", accounts.shape)
print("Transactions:", transactions.shape)
print("Loans:", loans.shape)
print("Customer Service:", customer_service.shape)

Customers: (5000, 26)
Accounts: (7500, 6)
Transactions: (100000, 7)
Loans: (2500, 7)
Customer Service: (10000, 6)


## 2. Aggregate Transactions

Aggregate transaction-level activity to the account level so transaction behaviour can be integrated without creating duplicate customer records.


In [3]:
transactions.columns.tolist()

['transaction_id',
 'customer_id',
 'account_id',
 'transaction_date',
 'transaction_type',
 'amount',
 'channel']

In [4]:
transaction_summary = (
    transactions
    .groupby("account_id")
    .agg(
        transaction_count=("transaction_id", "count"),
        total_transaction_amount=("amount", "sum"),
        average_transaction_amount=("amount", "mean")
    )
    .reset_index()
)

transaction_summary.head()

,account_id,transaction_count,total_transaction_amount,average_transaction_amount
0,A000001,21,5303.55,252.550000
1,A000002,6,2150.08,358.346667
2,A000003,2,596.06,298.030000
3,A000004,23,4566.33,198.536087
4,A000005,3,337.35,112.450000


## 3. Build Customer Account Summary

Combine account information with transaction activity and aggregate the results to one record per customer.


In [6]:
# Merge Accounts with Transaction Summary
accounts_with_transactions = accounts.merge(
    transaction_summary,
    on="account_id",
    how="left"
)

print("Shape:", accounts_with_transactions.shape)
accounts_with_transactions.head()

Shape: (7500, 9)


,account_id,customer_id,account_type,opening_date,status,balance,transaction_count,total_transaction_amount,average_transaction_amount
0,A000001,C00850,Savings,2021-11-10,Active,66553.46,21.0,5303.55,252.550000
1,A000002,C04939,Savings,2019-08-09,Active,46617.82,6.0,2150.08,358.346667
2,A000003,C02706,Savings,2023-05-04,Active,155264.40,2.0,596.06,298.030000
3,A000004,C02804,Current,2021-02-15,Active,135661.35,23.0,4566.33,198.536087
4,A000005,C04799,Savings,2019-03-26,Active,12340.66,3.0,337.35,112.450000


In [22]:
# Create Customer-Level Summary
customer_account_summary = (
    accounts_with_transactions
    .groupby("customer_id")
    .agg(
        total_accounts=("account_id", "nunique"),
        total_transaction_count=("transaction_count", "sum"),
        total_transaction_amount=("total_transaction_amount", "sum")
    )
    .reset_index()
)

customer_account_summary["average_transaction_amount"] = (
    customer_account_summary["total_transaction_amount"]
    / customer_account_summary["total_transaction_count"]
)

customer_account_summary.head()

,customer_id,total_accounts,total_transaction_count,total_transaction_amount,average_transaction_amount
0,C00001,4,22.0,5140.08,233.640000
1,C00002,1,21.0,6220.92,296.234286
2,C00003,2,16.0,4599.73,287.483125
3,C00004,1,15.0,3998.08,266.538667
4,C00005,2,19.0,7296.95,384.050000


## 4. Integrate Loan Information

Summarize loan activity at the customer level so loan information can be added to the Customer 360 dataset.


In [23]:
loans.columns.tolist()

['loan_id',
 'customer_id',
 'loan_type',
 'loan_amount',
 'interest_rate',
 'start_date',
 'repayment_status']

In [24]:
# Create customer-level loan summary
loan_summary = (
    loans
    .groupby("customer_id")
    .agg(
        total_loans=("loan_id", "nunique"),
        total_loan_amount=("loan_amount", "sum"),
        average_interest_rate=("interest_rate", "mean")
    )
    .reset_index()
)

print("Shape:", loan_summary.shape)
loan_summary.head()

Shape: (1962, 4)


,customer_id,total_loans,total_loan_amount,average_interest_rate
0,C00003,1,262227.49,9.120
1,C00006,1,162425.46,13.450
2,C00007,1,240496.42,13.620
3,C00010,2,127036.24,14.055
4,C00012,1,100559.94,11.110


In [25]:
# Add loans to our Customer 360 base
customer_account_loan_summary = customer_account_summary.merge(
    loan_summary,
    on="customer_id",
    how="left"
)

print("Shape:", customer_account_loan_summary.shape)
customer_account_loan_summary.head()

Shape: (3878, 8)


,customer_id,total_accounts,total_transaction_count,total_transaction_amount,average_transaction_amount,total_loans,total_loan_amount,average_interest_rate
0,C00001,4,22.0,5140.08,233.640000,NaN,NaN,NaN
1,C00002,1,21.0,6220.92,296.234286,NaN,NaN,NaN
2,C00003,2,16.0,4599.73,287.483125,1.0,262227.49,9.12
3,C00004,1,15.0,3998.08,266.538667,NaN,NaN,NaN
4,C00005,2,19.0,7296.95,384.050000,NaN,NaN,NaN


In [26]:
customer_service.columns.tolist()

['interaction_id',
 'customer_id',
 'interaction_date',
 'complaint_type',
 'resolution_status',
 'resolution_days']

## 5. Integrate Customer Service

Aggregate customer-service interactions to the customer level so service activity can be added to the Customer 360 dataset.


In [27]:
# Create service summary
service_summary = (
    customer_service
    .groupby("customer_id")
    .agg(
        total_service_interactions=("interaction_id", "count"),
        average_resolution_days=("resolution_days", "mean")
    )
    .reset_index()
)

print("Shape:", service_summary.shape)
service_summary.head()

Shape: (4303, 3)


,customer_id,total_service_interactions,average_resolution_days
0,C00001,1,8.0
1,C00002,2,11.5
2,C00003,1,6.0
3,C00004,1,2.0
4,C00005,1,8.0


In [28]:
# Add service information
customer_account_loan_service = customer_account_loan_summary.merge(
    service_summary,
    on="customer_id",
    how="left"
)

print("Shape:", customer_account_loan_service.shape)
customer_account_loan_service.head()

Shape: (3878, 10)


,customer_id,total_accounts,total_transaction_count,total_transaction_amount,average_transaction_amount,total_loans,total_loan_amount,average_interest_rate,total_service_interactions,average_resolution_days
0,C00001,4,22.0,5140.08,233.640000,NaN,NaN,NaN,1.0,8.0
1,C00002,1,21.0,6220.92,296.234286,NaN,NaN,NaN,2.0,11.5
2,C00003,2,16.0,4599.73,287.483125,1.0,262227.49,9.12,1.0,6.0
3,C00004,1,15.0,3998.08,266.538667,NaN,NaN,NaN,1.0,2.0
4,C00005,2,19.0,7296.95,384.050000,NaN,NaN,NaN,1.0,8.0


In [29]:
customer_account_loan_service.shape

(3878, 10)

In [30]:
customer_account_loan_service.columns.tolist()

['customer_id',
 'total_accounts',
 'total_transaction_count',
 'total_transaction_amount',
 'average_transaction_amount',
 'total_loans',
 'total_loan_amount',
 'average_interest_rate',
 'total_service_interactions',
 'average_resolution_days']

## 6. Create Customer 360 Dataset

Combine the master customer table with account, transaction, loan, and service metrics to create the unified Customer 360 analytical dataset.


In [31]:
# Merge Everything
customer_360 = customers.merge(
    customer_account_loan_service,
    on="customer_id",
    how="left"
)

print("Shape:", customer_360.shape)

Shape: (5000, 35)


In [32]:
# Inspect the result
customer_360.head()

,customer_id,age,gender,country,occupation,annual_income,credit_score,tenure_years,join_date,total_transactions,...,loan_to_income_ratio,total_accounts,total_transaction_count,total_transaction_amount,average_transaction_amount,total_loans,total_loan_amount_y,average_interest_rate,total_service_interactions,average_resolution_days
0,C00001,25,Male,France,Salaried,25700,577,7,2025-02-22,22,...,0.00,4.0,22.0,5140.08,233.640000,NaN,NaN,NaN,1.0,8.0
1,C00002,59,Male,Germany,Salaried,99800,682,12,2021-08-05,21,...,0.00,1.0,21.0,6220.92,296.234286,NaN,NaN,NaN,2.0,11.5
2,C00003,53,Female,Spain,Salaried,102200,786,11,2016-06-20,16,...,2.57,2.0,16.0,4599.73,287.483125,1.0,262227.49,9.12,1.0,6.0
3,C00004,42,Female,France,Self-Employed,48500,725,8,2016-11-17,15,...,0.00,1.0,15.0,3998.08,266.538667,NaN,NaN,NaN,1.0,2.0
4,C00005,42,Female,France,Self-Employed,75900,624,9,2023-07-22,19,...,0.00,2.0,19.0,7296.95,384.050000,NaN,NaN,NaN,1.0,8.0


In [33]:
customer_360.columns.tolist()

['customer_id',
 'age',
 'gender',
 'country',
 'occupation',
 'annual_income',
 'credit_score',
 'tenure_years',
 'join_date',
 'total_transactions',
 'total_transaction_value',
 'avg_transaction_value',
 'last_transaction_date',
 'loan_count',
 'total_loan_amount_x',
 'late_loan_count',
 'default_loan_count',
 'complaint_count',
 'escalated_count',
 'account_count',
 'total_balance',
 'active_account_count',
 'recency_days',
 'products_number',
 'churn',
 'loan_to_income_ratio',
 'total_accounts',
 'total_transaction_count',
 'total_transaction_amount',
 'average_transaction_amount',
 'total_loans',
 'total_loan_amount_y',
 'average_interest_rate',
 'total_service_interactions',
 'average_resolution_days']

In [34]:
customer_360.shape

(5000, 35)

In [35]:
print("Customers:", customer_360["customer_id"].nunique())
print("Rows:", len(customer_360))
print("Duplicate rows:", customer_360.duplicated().sum())

Customers: 5000
Rows: 5000
Duplicate rows: 0


In [36]:
# Inspect Overlapping metrics
customer_360[
    [
        "total_transactions",
        "total_transaction_count",
        "total_transaction_value",
        "total_transaction_amount",
        "avg_transaction_value",
        "average_transaction_amount",
        "loan_count",
        "total_loans",
        "total_loan_amount_x",
        "total_loan_amount_y",
        "complaint_count",
        "total_service_interactions",
        "account_count",
        "total_accounts"
    ]
].head()

,total_transactions,total_transaction_count,total_transaction_value,total_transaction_amount,avg_transaction_value,average_transaction_amount,loan_count,total_loans,total_loan_amount_x,total_loan_amount_y,complaint_count,total_service_interactions,account_count,total_accounts
0,22,22.0,5140.08,5140.08,233.640000,233.640000,0,NaN,0.00,NaN,1,1.0,4,4.0
1,21,21.0,6220.92,6220.92,296.234286,296.234286,0,NaN,0.00,NaN,2,2.0,1,1.0
2,16,16.0,4599.73,4599.73,287.483125,287.483125,1,1.0,262227.49,262227.49,1,1.0,2,2.0
3,15,15.0,3998.08,3998.08,266.538667,266.538667,0,NaN,0.00,NaN,1,1.0,1,1.0
4,19,19.0,7296.95,7296.95,384.050000,384.050000,0,NaN,0.00,NaN,1,1.0,2,2.0


In [37]:
customer_360_final = customer_360.drop(
    columns=[
        "total_transactions",
        "total_transaction_value",
        "avg_transaction_value",
        "loan_count",
        "total_loan_amount_x",
        "account_count"
    ]
).rename(
    columns={
        "total_transaction_count": "total_transactions",
        "total_transaction_amount": "total_transaction_value",
        "average_transaction_amount": "avg_transaction_value",
        "total_loans": "loan_count",
        "total_loan_amount_y": "total_loan_amount",
        "total_accounts": "account_count"
    }
)

print("Shape:", customer_360_final.shape)
print("\nColumns:")
print(customer_360_final.columns.tolist())

Shape: (5000, 29)

Columns:
['customer_id', 'age', 'gender', 'country', 'occupation', 'annual_income', 'credit_score', 'tenure_years', 'join_date', 'last_transaction_date', 'late_loan_count', 'default_loan_count', 'complaint_count', 'escalated_count', 'total_balance', 'active_account_count', 'recency_days', 'products_number', 'churn', 'loan_to_income_ratio', 'account_count', 'total_transactions', 'total_transaction_value', 'avg_transaction_value', 'loan_count', 'total_loan_amount', 'average_interest_rate', 'total_service_interactions', 'average_resolution_days']


In [38]:
validation = {
    "rows": len(customer_360_final),
    "columns": len(customer_360_final.columns),
    "unique_customers": customer_360_final["customer_id"].nunique(),
    "duplicate_customer_ids": customer_360_final["customer_id"].duplicated().sum()
}

pd.Series(validation)

rows                      5000
columns                     29
unique_customers          5000
duplicate_customer_ids       0
dtype: int64

In [39]:
# Final columns contains missing values and how many
missing_summary = (
    customer_360_final
    .isna()
    .sum()
    .sort_values(ascending=False)
)

missing_summary[missing_summary > 0]

total_loan_amount             3481
average_interest_rate         3481
loan_count                    3481
total_service_interactions    1667
average_resolution_days       1667
total_transactions            1122
avg_transaction_value         1122
account_count                 1122
total_transaction_value       1122
dtype: int64

In [41]:
zero_fill_columns = [
    "loan_count",
    "total_loan_amount",
    "average_interest_rate",
    "total_service_interactions",
    "average_resolution_days",
    "total_transactions",
    "avg_transaction_value",
    "account_count",
    "total_transaction_value"
]

customer_360_final[zero_fill_columns] = (
    customer_360_final[zero_fill_columns]
    .fillna(0)
)
customer_360_final[zero_fill_columns].isna().sum()

loan_count                    0
total_loan_amount             0
average_interest_rate         0
total_service_interactions    0
average_resolution_days       0
total_transactions            0
avg_transaction_value         0
account_count                 0
total_transaction_value       0
dtype: int64

In [42]:
print("Rows:", len(customer_360_final))
print("Columns:", len(customer_360_final.columns))
print("Unique customers:", customer_360_final["customer_id"].nunique())
print("Duplicate customer IDs:", customer_360_final["customer_id"].duplicated().sum())
print("Duplicate rows:", customer_360_final.duplicated().sum())
print("Total missing values:", customer_360_final.isna().sum().sum())

print("\nColumns with missing values:")
print(
    customer_360_final.isna().sum()
    .loc[lambda x: x > 0]
)

Rows: 5000
Columns: 29
Unique customers: 5000
Duplicate customer IDs: 0
Duplicate rows: 0
Total missing values: 0

Columns with missing values:
Series([], dtype: int64)


In [43]:
# Save final customer 360 dataset
from pathlib import Path

final_path = Path("../data/final")
final_path.mkdir(parents=True, exist_ok=True)

output_file = final_path / "customer_360.csv"

customer_360_final.to_csv(
    output_file,
    index=False
)

print(f"Saved successfully: {output_file}")
print(f"Rows: {len(customer_360_final):,}")
print(f"Columns: {len(customer_360_final.columns)}")

Saved successfully: ..\data\final\customer_360.csv
Rows: 5,000
Columns: 29
